In [1]:
import os

def show_inputs(depth=4, max_files_per_dir=6):
    """Print what is attached under /kaggle/input - check it before trusting the rest."""
    root0 = os.environ.get("SAT_INPUT", "/kaggle/input")
    for root, dirs, files in os.walk(root0):
        dirs[:] = sorted(d for d in dirs if not d.startswith("."))
        level = root.replace(root0, "").count(os.sep)
        if level > depth:
            dirs[:] = []
            continue
        print("  " * level + root + "/")
        for fn in sorted(files)[:max_files_per_dir]:
            print("  " * level + "  " + fn)
        if len(files) > max_files_per_dir:
            print("  " * level + f"  ... +{len(files) - max_files_per_dir} more files")

show_inputs()

/kaggle/input/
  /kaggle/input/datasets/
    /kaggle/input/datasets/ferdousjannatul/
      /kaggle/input/datasets/ferdousjannatul/sat-ext/
        /kaggle/input/datasets/ferdousjannatul/sat-ext/sat_ext/
          README.md
          analysis.py
          audit.py
          config.py
          dataset.py
          dota.py
          ... +10 more files
      /kaggle/input/datasets/ferdousjannatul/sat-results-08b/
        /kaggle/input/datasets/ferdousjannatul/sat-results-08b/data_sky_grouped/
          data.yaml
        /kaggle/input/datasets/ferdousjannatul/sat-results-08b/results/
        /kaggle/input/datasets/ferdousjannatul/sat-results-08b/results_08B/
      /kaggle/input/datasets/ferdousjannatul/sat-results-g/
        rtdetr-l.pt
        yolo11m.pt
        yolo11n.pt
        yolo11s.pt
        yolo26n.pt
        yolov8m.pt
        /kaggle/input/datasets/ferdousjannatul/sat-results-g/data_sky_grouped/
          data.yaml
        /kaggle/input/datasets/ferdousjannatul/sat-results-g/re

In [2]:
import os, sys, warnings
warnings.filterwarnings("ignore")
INPUT_ROOT = os.environ.get("SAT_INPUT", "/kaggle/input")

def _find_code_root():
    for root, dirs, files in os.walk(INPUT_ROOT):
        if {"config.py", "audit.py", "metrics.py", "infer.py"} <= set(files):
            return root
    return None

CODE_ROOT = os.environ.get("SAT_CODE") or _find_code_root()
assert CODE_ROOT, "sat_ext code folder not found - Add Input -> your 'sat-ext' code Dataset"
sys.path.insert(0, CODE_ROOT)
print("CODE_ROOT:", CODE_ROOT)

import numpy as np, pandas as pd
pd.set_option("display.width", 250); pd.set_option("display.max_columns", 60); pd.set_option("display.max_colwidth", 70)
import config as cfg, utils
utils.ensure_dirs(); utils.set_seed(cfg.SEED)
print("work dir:", cfg.WORK_DIR, "| smoke test:", cfg.SMOKE)

CODE_ROOT: /kaggle/input/datasets/ferdousjannatul/sat-ext/sat_ext
work dir: /kaggle/working | smoke test: False


In [3]:
import plans, train
PLAN, SLICES = plans.ABLATION["08B"], plans.ABLATION_SLICES["08B"]
print(len(PLAN), "runs:", PLAN, "| slices:", SLICES or "all of the split")
utils.require_gpu()
assert utils.install_detection_stack(), "pip install failed - is Internet ON?"
done = train.run_ablation(PLAN, SLICES)

5 runs: [('sky_grouped', 'YOLO11n', 42), ('sky_grouped', 'YOLO11s', 42), ('sky_grouped', 'YOLO11m', 42), ('sky_grouped', 'YOLOv8m', 42), ('sky_grouped', 'RT-DETR-L', 42)] | slices: [480, 320]
GPU: {'device': 'cuda:0', 'gpu': 'Tesla T4', 'n_gpu': 2, 'torch': '2.11.0+cu128'}
sky_grouped__YOLO11n__s42: 14 rows already done, running the 4 missing ones
materialized 'sky_grouped' (0b03f1f732a3) -> /kaggle/working/data_sky_grouped  images {'train': 922, 'val': 193, 'test': 192}
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
  sky_grouped__YOLO11n__s42 sahi480  paper_nms_iou_c25 AP=0.3109 AP50=0.7069 TP=2403 FP=4597 dupFP=109
  sky_grouped__YOLO11n__s42 sahi480  greedynmm_ios_c25 AP=0.1854 AP50=0.5579 TP=1991 FP=3622 dupFP=18
  sky_gr

In [4]:
import shutil
z = shutil.make_archive(os.path.join(cfg.WORK_DIR, "results_" + "08B"), "zip", cfg.RESULTS_DIR)
print("results zip:", z)
shutil.rmtree(os.path.join(cfg.WORK_DIR, "runs"), ignore_errors=True)

results zip: /kaggle/working/results_08B.zip
